In [26]:
import pandas as pd

In [31]:
ratings = pd.read_csv(
    "MovieLens 1M/ratings.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names = ['UserID','MovieID','Rating','Timestamp']
)

df = pd.read_csv('display_movie_data.csv')

Weighted Rating (IMDb style)

Use a combination of both rating and popularity.

### Formula

$$
WR = \frac{v}{v + m} \cdot R + \frac{m}{v + m} \cdot C
$$

---

### Meaning of terms

- **R** = Mean rating of the movie  
- **v** = Number of ratings (vote count)  
- **C** = Global average rating (across all movies)  
- **m** = Minimum votes threshold (e.g., 1000)

---

### Why this works

- High vote count → Trust the movie’s own rating more  
- Low vote count → Pull the rating toward global average  
- Prevents misleading cases like *"5-star rating with only 3 votes"*  

In [46]:
def get_popular_movies(display_df,ratings_df):
    df      = display_df.copy()
    ratings = ratings_df.copy()
    
    ratings_aggregate = ratings.groupby('MovieID')['Rating'].agg(['count', 'mean']).reset_index()
    
    df = df.merge(ratings_aggregate, on = 'MovieID')
                                                                    # keeping count first, as a popular movie maybe a somewhat less ratings than a unpopular good movie
                                                                    # eg:
                                                                    # Seven Samuai - Count: 628     Mean: 4.56
                                                                    # Godfather    - Count: 2200+   Mean: 4.55 
                                                                    # Godfather    - Count: 2200+   Mean: 4.2 
                                                                    
                                                                    
    # global average BEFORE heavy filtering
    C = df['mean'].mean()
    
    #light filtering
    df = df[(df['count'] > 200)]
        

    # dynamic threshold
    m = df['count'].quantile(0.75)


    df['weighted_rating'] = (
        (df['count'] / (df['count'] + m)) * df['mean'] +
        (m / (df['count'] + m)) * C
    )

    df_sorted = df.sort_values(by='weighted_rating', ascending=False)
        
    
    return df_sorted.head(25)

In [47]:
new_df = get_popular_movies(df,ratings)

In [50]:
new_df.head()

,MovieID,title,release_year,original_language,genre,overview,production_companies,keywords,cast,director,poster_url,count,mean,weighted_rating
307,318,The Shawshank Redemption,1994,en,"crime, drama",Imprisoned in the 1940s for the double murder ...,Castle Rock Entertainment,"prison, friendship, police brutality, corrupti...","Tim Robbins, Morgan Freeman, Bob Gunton, Willi...",Frank Darabont,https://image.tmdb.org/t/p/w500/9cqNxx0GxF0bfl...,2227,4.554558,4.223543
251,260,Star Wars,1977,en,"science fiction, adventure, sci-fi, fantasy, a...",Princess Leia is captured and held hostage by ...,"Lucasfilm Ltd., 20th Century Fox","empire, galaxy, rebellion, android, hermit, sm...","Mark Hamill, Harrison Ford, Carrie Fisher, Pet...",George Lucas,https://image.tmdb.org/t/p/w500/6FfCtAuVAW8XJj...,2991,4.453694,4.210502
797,858,The Godfather,1972,en,"crime, action, drama","Spanning the years 1945 to 1955, a chronicle o...","Paramount Pictures, Alfran Productions","based on novel or book, loss of loved one, lov...","Marlon Brando, Al Pacino, James Caan, Robert D...",Francis Ford Coppola,https://image.tmdb.org/t/p/w500/3bhkrj58Vtu7en...,2223,4.524966,4.200971
510,527,Schindler's List,1993,en,"war, history, drama",The true story of how businessman Oskar Schind...,Amblin Entertainment,"factory, hero, nazi, concentration camp, ss (n...","Liam Neeson, Ben Kingsley, Ralph Fiennes, Caro...",Steven Spielberg,https://image.tmdb.org/t/p/w500/sF1U4EUQS8YHUY...,2304,4.510417,4.198588
1103,1198,Raiders of the Lost Ark,1981,en,"action, adventure",When Dr. Indiana Jones – the tweed-suited prof...,"Paramount Pictures, Lucasfilm Ltd.","egypt, treasure, medallion, swastika, saving t...","Harrison Ford, Karen Allen, Paul Freeman, John...",Steven Spielberg,https://image.tmdb.org/t/p/w500/ceG9VzoRAVGwiv...,2514,4.477725,4.193471


In [51]:
new_df['director'].value_counts()

director
Steven Spielberg                   3
Francis Ford Coppola               2
Alfred Hitchcock                   2
Frank Darabont                     1
George Lucas                       1
Bryan Singer                       1
M. Night Shyamalan                 1
Sam Mendes                         1
Jonathan Demme                     1
Irvin Kershner                     1
Lana Wachowski, Lilly Wachowski    1
Michael Curtiz                     1
Rob Reiner                         1
Miloš Forman                       1
Joel Coen                          1
Stanley Kubrick                    1
Quentin Tarantino                  1
Mel Gibson                         1
Terry Gilliam, Terry Jones         1
Curtis Hanson                      1
Ridley Scott                       1
Name: count, dtype: int64

Steven Spielberg has 3 movies out 25 most popular movies

In [53]:
new_df['release_year'].value_counts()

release_year
1999    3
1994    2
1995    2
1974    2
1977    1
1972    1
1993    1
1981    1
1991    1
1998    1
1980    1
1942    1
1987    1
1975    1
1996    1
1963    1
1997    1
1982    1
1959    1
1954    1
Name: count, dtype: int64

In [54]:
new_df.to_csv('popular_movies.csv',index = False)

In [56]:
sample_movie = new_df.iloc[0,]

sample_movie

MovieID                                                               318
title                                            The Shawshank Redemption
release_year                                                         1994
original_language                                                      en
genre                                                        crime, drama
overview                Imprisoned in the 1940s for the double murder ...
production_companies                            Castle Rock Entertainment
keywords                prison, friendship, police brutality, corrupti...
cast                    Tim Robbins, Morgan Freeman, Bob Gunton, Willi...
director                                                   Frank Darabont
poster_url              https://image.tmdb.org/t/p/w500/9cqNxx0GxF0bfl...
count                                                                2227
mean                                                             4.554558
weighted_rating                       

In [57]:
sample_movie['title']

'The Shawshank Redemption'